# 03 · Pretraining by next-token prediction

**NVIDIA TFM Blueprint, notebook 3 of 5.** The model reads a history left to
right and, at every position, predicts the next token. Nobody labels anything:
the next token is the label. This notebook composes the model, trains it for a
few steps, and then reproduces the bug in v1's training, which made its
published model copy its input.

Every notebook runs on CPU in about a minute on a small generated book, so
you can read the whole series without downloading anything. To see real rows,
build the TabFormer splits once (notebook 01 shows how) and set
`TFM_TABFORMER_ROOT`; the notebooks then load 24 TabFormer users with
`config.small.yaml`. The full-size runs are recipe commands, listed in notebooks 03
and 05, not notebook cells.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from tfm import Binary, Decoder, EagerAttention, Flat, LastTokenPool, NextEvent, NoPE, TrxFM
from tfm.data.collate import minibatches
from tfm.runtime import build_model
from tfm.training.diagnostics import label_alignment_losses

from tfm.config import load_recipe_config
from tfm.data.adapters.paths import ADAPTER_ENV
from tfm.extras.capability import extra_available
from tfm.runtime import prepare_data
from tfm.runtime.common import recipe_dir
from tfm.training.reproducibility import seed_everything

USING_TABFORMER = bool(os.environ.get(ADAPTER_ENV["tabformer"]))
CONFIG = recipe_dir("blueprint_v1") / ("config.small.yaml" if USING_TABFORMER else "config.smoke.yaml")
config = load_recipe_config(CONFIG)
if config.model.backbone == "huggingface" and not extra_available("transformers"):
    print("transformers extra not installed; using the eager decoder (uv sync --extra huggingface)")
    composition = config.model.composition.model_copy(update={"position": "nope"})
    model_cfg = config.model.model_copy(update={"backbone": "eager", "composition": composition})
    config = config.model_copy(update={"model": model_cfg})
seed_everything(config.seed)
prepared = prepare_data(config)
tokenizer = prepared.tokenizer
train_batch, val_batch, test_batch = prepared.batches
print("config", CONFIG.name, "| adapter", config.data.adapter, "| backbone", config.model.backbone)

## 1. The model is five choices

`TrxFM(...)` names each design choice: a flat token sequence read by a causal
decoder, how positions are encoded, the attention kernel, next-token prediction
as the objective, and the last token's state as the embedding. The recipe's
YAML `model.composition` block serializes the same call.

v1's model is a HuggingFace Llama (29M parameters in v1; 25.9M here, without the
`CUST` vocabulary). Llama rotates queries and keys
by position inside attention (rotary positions, RoPE), so its recipe configs say
`position: rope`, and the kit refuses a Llama composition that claims no
positions. The kit's own eager decoder has none (`NoPE`), which is what the
generated book uses here.

In [ ]:
dec = config.model.composition.granularity.decoder
composed = TrxFM(
    granularity=Flat(
        Decoder(
            depth=dec.depth, mask=dec.mask, hidden_size=dec.hidden_size, num_heads=dec.num_heads,
            dropout=dec.dropout, vocab_size=tokenizer.vocab_size, max_seq_len=config.data.max_tokens,
        )
    ),
    position=NoPE(),
    attention=EagerAttention(),
    objective=NextEvent(),
    readout=LastTokenPool(),
    heads={"fraud": Binary()},
)
print("parameters", sum(p.numel() for p in composed.parameters()))
print("this config:", config.model.backbone, "backbone, position", config.model.composition.position)

## 2. A few training steps

`model(batch, compute_loss=True)` returns the next-token loss. The fraud
labels are not in it. The loss should fall steadily; on the generated book a
few dozen steps are enough to see that.

In [ ]:
STEPS = 60


def train(loss_fn):
    seed_everything(config.seed)
    model = build_model(config, tokenizer.vocab_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
    chunks = minibatches(train_batch, config.training.batch_size)
    losses = []
    for step in range(STEPS):
        batch = chunks[step % len(chunks)]
        loss = loss_fn(model, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return model.eval(), losses


def next_token_loss(model, batch):
    return model(batch, compute_loss=True).loss


model, losses = train(next_token_loss)
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("next-token loss")
plt.tight_layout()
plt.show()

## 3. The bug v1 shipped

To predict the next token, the logits at position *i* must be scored against
token *i + 1*. v1's data module set `labels = input_ids`, and its training loop
passed them to a loss that does not shift, so the logits at *i* were scored
against token *i*: the token the model has just read. A causal model can see
that token, so it learns to copy it, and its loss still falls nicely. The
published v1 checkpoint did exactly that
([ADR 0017](../../docs/architecture/0017-v1-blueprint-on-the-sdk.md)).

The fix is to shift the labels once: `labels[:n-1] = tokens[1:]`, the rest
ignored, with nothing downstream shifting again. The kit's `NextEvent` does
this. Train a second model with v1's unshifted loss, then score both on
validation histories both ways: against the next token (what the model should
predict) and against the same token (what a copying model predicts).

In [ ]:
def unshifted_loss(model, batch):
    logits = model(batch).logits
    targets = batch.input_ids.masked_fill(batch.attention_mask == 0, -100)
    return F.cross_entropy(logits.flatten(0, 1), targets.flatten(), ignore_index=-100)


copier, _ = train(unshifted_loss)
with torch.no_grad():
    rows = {
        name: label_alignment_losses(m(val_batch).logits, val_batch.input_ids, val_batch.attention_mask)
        for name, m in (("shifted (correct)", model), ("unshifted (v1)", copier))
    }
for name, r in rows.items():
    print(f"{name:18}  next-token acc {r.next_token_accuracy:.2f}  same-token acc {r.same_token_accuracy:.2f}  copies: {r.learned_to_copy}")

The unshifted model predicts the token it can already see and almost never the
next one. Its embeddings describe the transaction it just read, not what the
history suggests comes next.

> **What changed from v1**
>
> - **Shifted labels.** `NextEvent` scores position *i* against token *i + 1*,
>   and every training backend is tested for it.
> - **Declared positions.** The Llama composition says `position: rope`, and
>   `composition.attention` selects Llama's attention kernel (`eager` or `sdpa`).
> - **No NeMo.** Training runs on PyTorch, with a HuggingFace Llama as an
>   optional backbone and FSDP2 for several GPUs.

## Train it for real

The full-size run is a recipe command. It needs `uv sync --extra huggingface
--extra baselines` and the TabFormer splits from notebook 01.

```bash
# One GPU (sized on a DGX Spark): v1's Llama block and schedule, public data only.
uv run tfm report --recipe blueprint_v1 \
  --config recipes/blueprint_v1/config.yaml --device cuda \
  --run-dir outputs/blueprint_v1/gb10
```

`config.yaml` sets `training.log_every: 50`, so the run prints a line every 50
steps (loss, learning rate, gradient norm, tokens per second, and time left)
and writes every step's loss to `train_log.json` in the run directory.

For several GPUs, launch the same config with `torchrun` (`training.backend: ddp`
gives each GPU a replica), or shard it with `--config multi_gpu` (`fsdp2`); the
[recipe README](../../recipes/blueprint_v1/README.md) has the details.
On an unfamiliar machine, check
[hardware and runtimes](../../docs/reference/hardware-and-runtimes.md) first:
the Llama backbone needs Python headers for Triton.

## Takeaways

- Pretraining needs no labels: the next token is the target.
- Check label alignment, not just the loss curve. A copying model's loss falls
  too.

Code: `tfm.api`, `tfm.objectives.next_event`, `tfm.training.diagnostics`.

Next: [04 · Embeddings](04_inference_embedding_extraction.ipynb).